<a href="https://colab.research.google.com/github/blankqspace/homework_compling_course/blob/main/instruct_finetuning_homework_Artamonova.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание № 10. Генерация текста

### Задание 1 (8 баллов).


Возьмите Russian split из вот этого датасета - https://huggingface.co/datasets/CohereLabs/aya_collection_language_split
Он все равно очень большой, поэтому отфильтруйте его до какого-то небольшого рабочего подмножества (например, до 2000 примеров длиной меньше 300 токенов).
Найдите какую-нибудь языковую модель на huggingface, которая не была дообучена на инструкциях (base). Дообучите ее на получившемся датасете.
Возьмите любую задачу из Russian Superglue (например, вот эту - https://russiansuperglue.com/tasks/task_info/MuSeRC) и создайте из нее небольшой оценочный датасет. Формат должен подходить под получившуюся модель, поэтому если в тексте есть отдельно text и question, то вам понадобится их соединить в один промпт. Сделайте предсказания изначальной моделью и дообученой. Посчитайте какую-нибудь метрику качества или даже несколько (например, точное совпадение с правильными ответом + bleu score). Справляется ли дообученная модель лучше? Проанализируйте несколько предсказаний отдельно.
Вы можете использовать модель любого размера и любую технику дообучения (полное дообучение, LoRA или QLoRA)


(*Это задание сложнее предыдущих, поэтому не стесняйтесь задавать вопросы в чате или лично)


### Задание 2 (2 балла)
Два дополнительных балла можно получить если размер модели больше 3B.

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset("CohereLabs/aya_collection_language_split", "russian", split="train")
tokenizer = AutoTokenizer.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

russian/train-00000-of-00002.parquet:   0%|          | 0.00/704M [00:00<?, ?B/s]

russian/train-00001-of-00002.parquet:   0%|          | 0.00/739M [00:00<?, ?B/s]

russian/validation-00000-of-00001.parque(…):   0%|          | 0.00/127M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

In [4]:
def filter_fn(batch):
    texts = batch["inputs"]
    tokens = tokenizer(texts, truncation=False)
    lengths = [len(ids) for ids in tokens["input_ids"]]
    return [l < 300 for l in lengths]

dataset = dataset.filter(filter_fn, batched=True, batch_size=5000)
dataset = dataset.shuffle(seed=42).select(range(2000))

Filter:   0%|          | 0/4005166 [00:00<?, ? examples/s]

In [5]:
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [7]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

pytorch_model.bin:   0%|          | 0.00/551M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/551M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3small_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [8]:
print_trainable_parameters(model)

trainable params: 4718592 || all params: 168552960 || trainable%: 2.7994714539572607


In [34]:
def format_example(example):
    return {
        "text": example["inputs"] + "\n" + example["targets"]
    }
dataset = dataset.map(format_example)
dataset = dataset.remove_columns([col for col in dataset.column_names if col != "text"])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [35]:
dataset[0]

{'text': 'Write a continuation for this paragraph - В последующие годы Влашич стал сильнейшим югославским десятиборцем, выиграв пять подряд национальных чемпионатов с 1979 по 1983 год. В 1979 году Влашич завоевал бронзовую медаль на проходивших в его родном Сплите Средиземноморских играх. На следующих играх в Касабланке в 1983 году\nон завоевал золото. В том же году у Влашича родилась дочь Бланка, названная так в честь города, в котором прошли игры. В 1983 году он также принимал участие в чемпионате мира в Хельсинки, где занял 16 место.'}

In [10]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 17.1 MB/s eta 0:00:00


In [39]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_steps=100,
        max_steps=600,
        learning_rate=5e-4,
        bf16=True,
        logging_steps=100,
        output_dir='outputs',
        max_length=512,
        max_grad_norm=1.0,
        neftune_noise_alpha=5,
        dataset_text_field="text",
    ),
)

In [40]:
model.config.use_cache = False
trainer.train()

Step,Training Loss
100,2.643268
200,2.583857
300,2.506392
400,2.457168
500,2.362845
600,2.283470


TrainOutput(global_step=600, training_loss=2.4728333536783853, metrics={'train_runtime': 691.6228, 'train_samples_per_second': 13.88, 'train_steps_per_second': 0.868, 'total_flos': 1034877499490304.0, 'train_loss': 2.4728333536783853})

In [41]:
model.save_pretrained('lora')

In [3]:
!pip install sacrebleu
import torch
import pandas as pd
import re
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import sacrebleu

In [1]:
!pip install -U bitsandbytes>=0.46.1

In [1]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "ai-forever/rugpt3small_based_on_gpt2"
LORA_PATH = "lora"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

print("Загрузка базовой модели...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    torch_dtype="auto",
    device_map="auto",
    cache_dir="./models",
    tie_word_embeddings=False
)

print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Применение адаптера из {LORA_PATH}...")
import os
if not os.path.exists(LORA_PATH):
    raise FileNotFoundError(f"Адаптер не найден: {LORA_PATH}\n Проверьте: {os.listdir('.')}")

lora_model = PeftModel.from_pretrained(base_model, LORA_PATH)
lora_model = lora_model.eval()

print(f"Тип: {type(lora_model)}")
print(f"PeftModel: {isinstance(lora_model, PeftModel)}")

Загрузка базовой модели...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3small_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Загрузка токенизатора...
Применение адаптера из lora...
Тип: <class 'peft.peft_model.PeftModelForCausalLM'>
PeftModel: True


In [2]:
print("\nТест генерации...")
test_prompt = "Write a continuation for this paragraph - Солнце светит ярко.\n"
inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=200).to(DEVICE)

with torch.no_grad():
    outputs = lora_model.generate(**inputs, max_new_tokens=30, pad_token_id=tokenizer.pad_token_id)

generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
print(f"Промпт: {test_prompt.strip()}")
print(f"Ответ: \"{generated}\"")


Тест генерации...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Промпт: Write a continuation for this paragraph - Солнце светит ярко.
Ответ: "Конечно, более сложная версия предложения - "Солнце светит ярко, когда оно находится в зените, и когда оно находится в зените"."


In [4]:
def load_muserc_simple(zip_path, n_samples=50):
    examples = []
    if not os.path.exists(zip_path):
        for root, dirs, files in os.walk("."):
            if zip_path in files:
                zip_path = os.path.join(root, zip_path)
                break

    with zipfile.ZipFile(zip_path, 'r') as zf:
        fname = next((f for f in zf.namelist() if f.endswith('test.jsonl')), None)
        if not fname:
            raise FileNotFoundError("test.jsonl не найден в архиве")
        with zf.open(fname) as f:
            for i, line in enumerate(f):
                if len(examples) >= n_samples: break
                if not line.strip(): continue
                rec = json.loads(line.decode('utf-8'))
                passage_text = rec['passage']['text'] if isinstance(rec.get('passage'), dict) else rec.get('passage', '')
                questions = rec['passage']['questions'] if isinstance(rec.get('passage'), dict) else rec.get('questions', [])
                for q in questions:
                    answers = q.get('answers', [])
                    options = [a.get('text', '') if isinstance(a, dict) else str(a) for a in answers]
                    label_idx = q.get('idx', 0)
                    correct_idx = 0
                    for a_idx, a in enumerate(answers):
                        if isinstance(a, dict) and a.get('idx') == label_idx:
                            correct_idx = a_idx
                            break
                    examples.append({
                        'paragraph': passage_text,
                        'question': q.get('question', ''),
                        'options': options,
                        'label': correct_idx,
                        'correct_answer_text': options[correct_idx] if options else ''
                    })
    print(f"Загружено {len(examples)} вопросов")
    return Dataset.from_list(examples)

ds = load_muserc_simple("MuSeRC.zip", n_samples=50)

Загружено 56 вопросов


In [5]:
def make_prompt(ex):
    opts = '\n'.join([f"{i+1}) {o}" for i,o in enumerate(ex.get('options', []))])
    return f"""Текст: {ex['paragraph'][:600]}

Вопрос: {ex['question']}

Варианты:
{opts}

Какой вариант правильный? Ответьте текстом варианта."""

In [6]:
def generate(model, tok, prompt, max_new=50):
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new, pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id, do_sample=True, temperature=0.1)
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    if len(generated) == 0: return "[NO_OUTPUT]"
    text = tok.decode(generated, skip_special_tokens=True).strip()
    return text if text else "[EMPTY]"

def exact_match_char(pred, ref):
    if not pred or not ref: return 0.0
    return 1.0 if ' '.join(pred.lower().split()) == ' '.join(ref.lower().split()) else 0.0

def compute_bleu(pred, ref):
    if not pred or not ref: return 0.0
    score = sacrebleu.sentence_bleu(pred, [ref], lowercase=True, tokenize='13a', smooth_method='exp')
    return score.score / 100.0

def evaluate_textual(model, tok, dataset, label="Model"):
    if not dataset or len(dataset)==0: return {'em':0, 'bleu':0, 'empty_rate':0}, pd.DataFrame()
    results = []
    print(f"Оценка {label} ({len(dataset)} примеров)...")
    empty_count = 0
    for ex in dataset:
        correct_text = ex.get('correct_answer_text', '')
        raw_output = generate(model, tok, make_prompt(ex), max_new=50)
        if raw_output in ["[NO_OUTPUT]", "[EMPTY]", ""]: empty_count += 1
        em = exact_match_char(raw_output, correct_text) if raw_output not in ["[NO_OUTPUT]", "[EMPTY]", ""] else 0.0
        bleu = compute_bleu(raw_output, correct_text) if raw_output not in ["[NO_OUTPUT]", "[EMPTY]", ""] else 0.0
        results.append({'question': ex['question'][:100], 'correct_text': correct_text, 'raw_output': raw_output, 'em': em, 'bleu': bleu, 'model': label})
    df = pd.DataFrame(results)
    return {'model': label, 'n': len(df), 'em': df['em'].mean(), 'bleu': df['bleu'].mean(), 'empty_rate': empty_count/len(df)*100}, df

In [9]:
import sacrebleu
import pandas as pd
print("\n" + "="*80)
print("НАЧАЛО ОЦЕНКИ")
print("="*80)

print("\nБазовая модель...")
base_model.eval()
m1, df1 = evaluate_textual(base_model, tokenizer, ds, "Base")

print("\nLoRA модель...")
m2, df2 = evaluate_textual(lora_model, tokenizer, ds, "LoRA")

print("\n" + "="*80)
print("РЕЗУЛЬТАТЫ")
print("="*80)
summary = pd.DataFrame([m1, m2])
print(summary[['model', 'n', 'em', 'bleu', 'empty_rate']].to_string(index=False))

print(f"\nРазница (LoRA - Base):")
print(f"   Exact Match: {(m2['em']-m1['em']):+.3f}")
print(f"   BLEU:        {(m2['bleu']-m1['bleu']):+.3f}")

df1.to_csv("results_base.csv", index=False, encoding='utf-8-sig')
df2.to_csv("results_lora.csv", index=False, encoding='utf-8-sig')
print("\nРезультаты сохранены!")


НАЧАЛО ОЦЕНКИ

Базовая модель...
Оценка Base (56 примеров)...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")



LoRA модель...
Оценка LoRA (56 примеров)...

РЕЗУЛЬТАТЫ
model  n  em     bleu  empty_rate
 Base 56 0.0 0.064277   10.714286
 LoRA 56 0.0 0.061977   10.714286

Разница (LoRA - Base):
   Exact Match: +0.000
   BLEU:        -0.002

Результаты сохранены!


In [11]:
base = pd.read_csv('results_base.csv')
lora = pd.read_csv('results_lora.csv')

In [12]:
base

,question,correct_text,raw_output,em,bleu,model
0,О чем дневник Анны Франк?,Дневник об отметках Анны.,Шаг за шагом Анна Франк описывает свою жизнь и...,0.0,0.015062,Base
1,Каким будет приложение Anne Frank App?,Приложение будет платным.,[EMPTY],0.0,0.000000,Base
2,Какая информация войдет в новой мобильное прил...,Карта Нидерландов.,"Варианты - Да, конечно.",0.0,0.081167,Base
3,Где скрывались члены семьи Франк и другие евреи?,В концлагере.,"Сначала напишите текст, который будет соответс...",0.0,0.013356,Base
4,"Как Анна называла место, где она и ее семья ск...",Она называла его Убежищем.,[EMPTY],0.0,0.000000,Base
5,"Как будет называться приложение, посвященное А...",Отто Франк.,"Сначала напишите текст, содержащий ответ на эт...",0.0,0.045739,Base
6,Когда в первый раз была опубликована книга с з...,В 1946 году.,В 1946 году Анна Франк была издана в Великобри...,0.0,0.072643,Base
7,На сколько языков переведён дневник?,Более чем на 60 языков.,[EMPTY],0.0,0.000000,Base
8,"Кто озвучивал фрагменты книжки с записями, впе...",Хелена Бонэм Картер.,"Варианты - Да, конечно. - Нет.",0.0,0.047677,Base
9,"Где родилась автор книги, на основе которой из...",Девочка родилась в Нидерландах.,"Варианты - А. Франк, проект, который был запущ...",0.0,0.017912,Base


In [13]:
lora

,question,correct_text,raw_output,em,bleu,model
0,О чем дневник Анны Франк?,Дневник об отметках Анны.,"Сначала напишите короткий абзац: ""Анна Франк б...",0.0,0.008130,LoRA
1,Каким будет приложение Anne Frank App?,Приложение будет платным.,[EMPTY],0.0,0.000000,LoRA
2,Какая информация войдет в новой мобильное прил...,Карта Нидерландов.,"В приложении будут фотографии и видеозаписи, д...",0.0,0.011231,LoRA
3,Где скрывались члены семьи Франк и другие евреи?,В концлагере.,В каком предложении нет подлежащего и сказуемо...,0.0,0.012193,LoRA
4,"Как Анна называла место, где она и ее семья ск...",Она называла его Убежищем.,[EMPTY],0.0,0.000000,LoRA
5,"Как будет называться приложение, посвященное А...",Отто Франк.,[EMPTY],0.0,0.000000,LoRA
6,Когда в первый раз была опубликована книга с з...,В 1946 году.,В 1946 году Анна Франк была издана в Англии.,0.0,0.167845,LoRA
7,На сколько языков переведён дневник?,Более чем на 60 языков.,[EMPTY],0.0,0.000000,LoRA
8,"Кто озвучивал фрагменты книжки с записями, впе...",Хелена Бонэм Картер.,"Варианты - Да, конечно. - Нет.",0.0,0.047677,LoRA
9,"Где родилась автор книги, на основе которой из...",Девочка родилась в Нидерландах.,"Варианты - А. Франк, А. Бонэм Картер, Хелен Бо...",0.0,0.007773,LoRA


LoRA ухудшила качество модели (которое итак было очень низкое), метрики плохие в обоих вариантах, но при использовании адаптора модель, как будто, "тупеет". Я могу предположить, что это может быть связано с неудачным выбором формата данных для обучения, я прочитала, что подавать не темплейт такой модели удобнее, чем уже обновленный формат, но возможно я прогадала и поэтому результаты получились такими низкими